# Phân loại ảnh bằng SIFT + Bag of Visual Words + SVM

Notebook này được thiết kế để tải mã nguồn từ GitHub và chạy trực tiếp trên Kaggle.

Trước khi chọn **Run All**:

1. Bật Internet trong phần **Notebook options** để Git và pip có thể hoạt động.
2. Chọn **Add Input** và gắn bộ dữ liệu Caltech-101 vào notebook.
3. Nếu repository là private, cần đổi `REPO_URL` sang URL có thông tin xác thực phù hợp. Không ghi token trực tiếp vào notebook công khai.

## 1. Cấu hình

In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/ngvihoa/bovw-image-classification.git"
BRANCH = "main"
PROJECT_DIR = Path("/kaggle/working/bovw-image-classification")
OUTPUT_DIR = Path("/kaggle/working/bovw_output")

# Để None, chương trình sẽ tự tìm thư mục chứa năm lớp trong /kaggle/input.
# Nếu tự động tìm không đúng, nhập đường dẫn cụ thể, ví dụ:
# DATA_DIR = Path("/kaggle/input/caltech101/101_ObjectCategories")
DATA_DIR = None

CLASSES = ["airplanes", "Motorbikes", "Faces", "watch", "car_side"]
VOCAB_SIZES = [50, 100, 200, 500]
SEED = 42

print("Project directory:", PROJECT_DIR)
print("Output directory :", OUTPUT_DIR)

## 2. Clone hoặc cập nhật repository

Ở lần chạy đầu tiên, cell này dùng `git clone`. Nếu thư mục dự án đã tồn tại do chạy lại notebook, cell sẽ dùng `git pull --ff-only`.

In [ ]:
import subprocess

def run_command(command, cwd=None):
    print("$", " ".join(map(str, command)))
    subprocess.run([str(item) for item in command], cwd=cwd, check=True)

if (PROJECT_DIR / ".git").is_dir():
    run_command(["git", "fetch", "origin", BRANCH], cwd=PROJECT_DIR)
    run_command(["git", "checkout", BRANCH], cwd=PROJECT_DIR)
    run_command(["git", "pull", "--ff-only", "origin", BRANCH], cwd=PROJECT_DIR)
else:
    run_command(["git", "clone", "--branch", BRANCH, "--single-branch", REPO_URL, PROJECT_DIR])

run_command(["git", "log", "-1", "--oneline"], cwd=PROJECT_DIR)

## 3. Cài đặt thư viện

Kaggle thường đã có NumPy, scikit-learn và OpenCV. Cell này vẫn kiểm tra file `requirements.txt` của repository và cài các gói còn thiếu hoặc chưa đúng phiên bản.

In [ ]:
import sys

requirements_file = PROJECT_DIR / "requirements.txt"
if not requirements_file.is_file():
    raise FileNotFoundError(f"Không tìm thấy {requirements_file}")

run_command([sys.executable, "-m", "pip", "install", "-q", "-r", requirements_file])

## 4. Kiểm tra dữ liệu Kaggle

In [ ]:
import os

input_root = Path("/kaggle/input")
print("Các dataset đã gắn vào notebook:")
for child in sorted(input_root.iterdir()) if input_root.exists() else []:
    print(" -", child)

if DATA_DIR is not None:
    missing_classes = [name for name in CLASSES if not (DATA_DIR / name).is_dir()]
    if missing_classes:
        raise FileNotFoundError(
            f"DATA_DIR={DATA_DIR} thiếu các thư mục lớp: {missing_classes}"
        )
    print("Sử dụng dữ liệu tại:", DATA_DIR)
else:
    print("DATA_DIR=None: script huấn luyện sẽ tự động dò dữ liệu.")

## 5. Huấn luyện và đánh giá

Cell này gọi script huấn luyện vừa được tải từ repository. Để chạy nhanh một cấu hình, có thể đổi `VOCAB_SIZES` thành `[500]`.

In [ ]:
training_script = PROJECT_DIR / "scripts" / "kaggle_train.py"
if not training_script.is_file():
    raise FileNotFoundError(
        f"Không tìm thấy {training_script}. Hãy bảo đảm file scripts/kaggle_train.py "
        "đã được commit và push lên GitHub trước khi chạy notebook."
    )

command = [
    sys.executable,
    training_script,
    "--output-dir",
    OUTPUT_DIR,
    "--seed",
    SEED,
    "--classes",
    *CLASSES,
    "--vocab-sizes",
    *VOCAB_SIZES,
]
if DATA_DIR is not None:
    command.extend(["--data-dir", DATA_DIR])

run_command(command, cwd=PROJECT_DIR)

## 6. Xem kết quả

In [ ]:
import pandas as pd
from IPython.display import display

summary_path = OUTPUT_DIR / "summary.csv"
summary = pd.read_csv(summary_path).sort_values("vocab_size")
display(summary.style.format({
    "accuracy": "{:.4f}",
    "precision_macro": "{:.4f}",
    "recall_macro": "{:.4f}",
    "f1_macro": "{:.4f}",
    "elapsed_seconds": "{:.1f}",
}))

best_row = summary.loc[summary["f1_macro"].idxmax()]
best_k = int(best_row["vocab_size"])
print(f"Cấu hình tốt nhất theo F1 macro: K={best_k}")

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

confusion_path = OUTPUT_DIR / f"k{best_k}" / "confusion_matrix.png"
image = Image.open(confusion_path)
plt.figure(figsize=(10, 8))
plt.imshow(image)
plt.axis("off")
plt.title(f"Ma trận nhầm lẫn của cấu hình K={best_k}")
plt.show()

## 7. Đóng gói kết quả để tải xuống

File ZIP sẽ xuất hiện trong mục **Output** của Kaggle sau khi notebook chạy xong.

In [ ]:
import shutil

archive_path = shutil.make_archive(
    "/kaggle/working/bovw_results", "zip", root_dir=OUTPUT_DIR
)
print("Đã tạo:", archive_path)